# Feedback loop: do escalated receipts make a better student?

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --no-deps git+https://github.com/sourav-ydv/receipt-router.git

## Settings

In [ ]:
import gc
import json
import random
import time

DATA_DIR = "/kaggle/input/datasets/souravydv25/receipt-sft"
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ARM = "escalation"
SEED = 42
INITIAL = 150
ROUNDS = 4
EPOCHS = 3
MAX_LEN = 2048

## Data

In [ ]:
from receipts.feedback import choose_additions, split_stream


def read_jsonl(name):
    with open(f"{DATA_DIR}/{name}", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


pool = {row["id"]: row for row in read_jsonl("train_pool.jsonl")}
test = [row for row in read_jsonl("test.jsonl") if row["usable"]]

initial, chunks = split_stream(list(pool), INITIAL, ROUNDS, SEED)
selected = [i for i in initial if pool[i]["ok"]]
good = [i for i in pool if pool[i]["ok"]]

print(len(pool), "labeled,", len(good), "usable labels,", len(test), "test receipts")
print("start with", len(selected), "examples; chunks of", [len(c) for c in chunks])

## Training and answering

In [ ]:
import torch
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

from receipts.formatting import INSTRUCTION, make_example
from receipts.schema import Receipt
from receipts.scoring import score, summarize
from receipts.validators import check, parse_receipt


def train(ids):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL,
        max_seq_length=MAX_LEN,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        lora_alpha=32,
        lora_dropout=0,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )
    rows = []
    for i in ids:
        example = make_example(pool[i]["text"], Receipt(**pool[i]["label"]))
        rows.append({"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)})
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=Dataset.from_list(rows),
        args=SFTConfig(
            dataset_text_field="text",
            max_seq_length=MAX_LEN,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=8,
            num_train_epochs=EPOCHS,
            learning_rate=2e-4,
            lr_scheduler_type="cosine",
            warmup_ratio=0.05,
            weight_decay=0.01,
            optim="adamw_8bit",
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            logging_steps=20,
            eval_strategy="no",
            save_strategy="no",
            seed=SEED,
            report_to="none",
            output_dir="outputs",
        ),
    )
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )
    trainer.train()
    return model, tokenizer


def answers(model, tokenizer, texts, batch_size=4):
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    outputs = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        prompts = [
            tokenizer.apply_chat_template(
                [
                    {"role": "system", "content": INSTRUCTION},
                    {"role": "user", "content": text},
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
            for text in batch
        ]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            generated = model.generate(**inputs, max_new_tokens=800, do_sample=False)
        new_tokens = generated[:, inputs["input_ids"].shape[1]:]
        outputs.extend(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))
    return outputs


def problems_of(raw):
    pred = parse_receipt(raw)
    return pred, (["unparseable"] if pred is None else check(pred))

## Run the rounds

In [ ]:
history = []

for round_number in range(ROUNDS + 1):
    started = time.time()
    model, tokenizer = train(selected)

    raw = answers(model, tokenizer, [row["text"] for row in test])
    results = []
    accepted = []
    for row, text in zip(test, raw):
        pred, problems = problems_of(text)
        result = score(pred or Receipt(), Receipt(**row["label"]))
        results.append(result)
        if not problems:
            accepted.append(result)

    overall = summarize(results)
    record = {
        "round": round_number,
        "train_size": len(selected),
        "test_exact": overall["exact"],
        "test_items_f1": overall["items_f1"],
        "test_accepted": len(accepted) / len(results),
        "test_exact_accepted": summarize(accepted)["exact"] if accepted else 0.0,
    }

    if round_number < ROUNDS:
        chunk = chunks[round_number]
        chunk_raw = answers(model, tokenizer, [pool[i]["text"] for i in chunk])
        failed = [i for i, text in zip(chunk, chunk_raw) if problems_of(text)[1]]
        added = choose_additions(ARM, chunk, failed, good, SEED + round_number)
        selected = selected + added
        record["chunk_size"] = len(chunk)
        record["chunk_escalation_rate"] = len(failed) / len(chunk)
        record["added"] = len(added)

    record["minutes"] = round((time.time() - started) / 60, 1)
    history.append(record)
    print(record)

    with open(f"/kaggle/working/feedback_{ARM}_seed{SEED}.json", "w") as f:
        json.dump(history, f, indent=2)

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

## Summary

In [ ]:
print("round  train  exact  items_f1  accepted  escalation_on_next_chunk")
for r in history:
    print(
        r["round"],
        r["train_size"],
        f"{r['test_exact']:.1%}",
        f"{r['test_items_f1']:.1%}",
        f"{r['test_accepted']:.1%}",
        f"{r.get('chunk_escalation_rate', float('nan')):.1%}",
    )